# LexVerify Local LLM Fine-Tuning

This notebook fine-tunes **Llama 3.2 3B** on the LexVerify legal corpus using **Unsloth** and **QLoRA**. 
It is optimized for Google Colab's free T4 GPU (16GB VRAM) and will export a GGUF file suitable for 8GB RAM laptops.

### Instructions
1. Upload your `data/training_data.jsonl` to the Colab files pane.
2. Run all cells.
3. Download the generated `lexverify-legal-3b.Q4_K_M.gguf` file.
4. Place it in the `ollama/` folder of the LexVerify project.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None # None for auto detection
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = "" # Add your HF token if required, not needed for base Llama 3.2
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,  
    loftq_config = None,
)

In [ ]:
chat_template = """<|start_header_id|>system<|end_header_id|>

You are a senior legal analyst for LexVerify, a verification-first legal AI system. Answer strictly using the provided context.<|eot_id|><|start_header_id|>user<|end_header_id|>

{instruction}
{input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{output}<|eot_id|>"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        text = chat_template.format(instruction=instruction, input=input, output=output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

In [ ]:
import os
from datasets import load_dataset

if not os.path.exists("training_data.jsonl"):
    print("\n=======================================================")
    print("⚠️ PLEASE UPLOAD training_data.jsonl TO COLAB NOW")
    print("=======================================================\n")
else:
    dataset = load_dataset("json", data_files="training_data.jsonl", split="train")
    dataset = dataset.map(formatting_prompts_func, batched = True,)
    print(f"Loaded {len(dataset)} examples")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
model.save_pretrained("lexverify_lora_model")
tokenizer.save_pretrained("lexverify_lora_model")

In [ ]:
# Export to GGUF format
model.save_pretrained_gguf("lexverify-legal-3b", tokenizer, quantization_method="q4_k_m")
print("GGUF complete.")

# Mount Google Drive to save the model safely
try:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    print("Copying model to Google Drive (this may take a few minutes)...")
    shutil.copy("lexverify-legal-3b-unsloth.Q4_K_M.gguf", "/content/drive/MyDrive/lexverify-legal-3b-unsloth.Q4_K_M.gguf")
    print("✅ Model successfully saved to your Google Drive! You can download it from there.")
except Exception as e:
    print("Google Drive mount skipped or failed. Download lexverify-legal-3b-unsloth.Q4_K_M.gguf from the Colab files pane.")
